# ML-08 — Capstone Modeling Lane

Trains and compares Logistic Regression, Decision Tree, and Random Forest models on content refresh prioritization.


## 1. Method choice and why

We selected Random Forest Classifier as our primary model because it handles non-linear interactions between search volume, impressions, position, and content age without assuming linearity.


In [ ]:
import json
import pandas as pd

with open("outputs/model_results.json") as f:
    results = json.load(f)

print("Best model:", results["best_model"]["name"])
print("Selection metric:", results["best_model"]["selection_metric"])


## 2. Split design

We use **client_holdout** validation: entire pseudonymized clients are isolated into train and test splits so pages from the same client never appear in both.


In [ ]:
print("Split strategy:", results["split_strategy"])
print("Train rows:", results["train_rows"])
print("Test rows:", results["test_rows"])


## 3. Train + compare vs my baseline

Model comparison under client_holdout split:


In [ ]:
models_data = []
for name, m in results["models"].items():
    models_data.append({
        "Model": name,
        "ROC AUC": m["roc_auc"],
        "Avg Precision": m["average_precision"],
        "Precision@50": m["precision_at_50"],
        "Recall": m["recall"],
        "F1": m["f1"]
    })
base = results["baseline"]
models_data.append({
    "Model": "baseline_rules",
    "ROC AUC": base["baseline_roc_auc"],
    "Avg Precision": base["baseline_average_precision"],
    "Precision@50": base["baseline_precision_at_50"],
    "Recall": base["baseline_recall"],
    "F1": base["baseline_f1"]
})

comp_df = pd.DataFrame(models_data)
print(comp_df.to_string(index=False))


## 4. Errors and interpretation

Top features identified by Random Forest: `days_with_impressions` (16.06%), `log_impressions_90d` (12.85%), `avg_position` (10.84%), and `content_age_days` (9.50%).


In [ ]:
top_feats = pd.DataFrame(results["best_model"]["feature_importance_top"]).head(10)
print("Top 10 Feature Importances:")
print(top_feats.to_string(index=False))
